<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/dev/notebooks/04_phase3/DeiTTiny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Land Cover Regression – DeiT Tiny Experiments

This notebook implements a multimodal regression framework that combines image features with textual descriptions to predict land cover distributions.

The experiments are designed to evaluate the contribution of textual information to visual models and the effect of different caption types listed in the following:
  - Text-only captions
  - Hybrid captions
  - Vision-generated captions

The impact of different lightweight text encoders, (MiniLM,DistilBERT, RemoteCLIP) are also observed.

The image backbone used in this notebook is Deit Tiny, which is fine-tuned during training. Text encoders are kept frozen, and features are combined using a gated fusion mechanism.

All experiments are run under a consistent setup (same split, seed, training configuration) to ensure fair comparison.

## Environment Setup

This section installs the required libraries for the experiments.
These packages provide the core functionality for building and training the multimodal model.

In [1]:
!pip install -q timm transformers wandb open_clip_torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.4 MB/s eta 0:00:00


## Data Loading and Setup

The dataset is loaded from Google Drive and extracted locally. This setup ensures that images and their corresponding textual descriptions can be accessed efficiently during training.

In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

from torchvision import transforms

import wandb
import open_clip
from huggingface_hub import hf_hub_download
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
os.makedirs("/content/drive/MyDrive/2444347_DI725_term_project_phase3_predictions", exist_ok=True)

In [5]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [6]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV: /content/DI725_project_dataset_nomask/captions.csv
Images: /content/DI725_project_dataset_nomask/images
Number of images: 10000


## Experiment Configuration

This cell defines all experimental settings used throughout the notebook.
Hyperparameters include sample size, image size, batch size, number of epochs,
learning rate, weight decay, gradient clipping, and early stopping patience.

The DataLoader is configured via num_workers. The Dataset uses max_token_len
to control tokenization length. The regression head is configured via
regressor_hidden_dim and regressor_dropout. W&B project names and the
predictions output path are also centralized here to avoid hardcoding.

The target variables are the seven classes: Tree, Shrub, Grass, Crop,
Built-up, Barren, and Water. The text_scale parameter controls the strength of
the textual contribution during gated fusion. The random seed is set here and
applied globally for reproducibility.

In [7]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 224,

    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-4,
    "weight_decay": 1e-4,

    "model_name": "deit_tiny_patch16_224",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42,

    "text_scale": 0.7,

    # DataLoader
    "num_workers": 2,

    # Dataset
    "max_token_len": 128,

    # Model
    "regressor_hidden_dim": 256,
    "regressor_dropout": 0.25,

    # W&B
    "wandb_project": "DI725-Project_phase3_experiments",
    "wandb_project_scale": "DI725-Project_phase3_text_scale_exp",

    # Predictions output path
    "predictions_dir": "/content/drive/MyDrive/2444347_DI725_term_project_phase3_predictions",
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

## Reproducibility Setup

This cell fixes the random seed across Python, NumPy, and PyTorch to improve reproducibility. Using the same seed helps ensure that sampling, data splitting, and model training are comparable across experiments.

In [8]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])

## Data Loading and Text Cleaning

This section loads the caption and target data from the CSV file and prepares the text columns used in the multimodal experiments.

A cleaning function removes percentages, standalone numbers, and leakage related phrases from captions. This step prevents the model from directly using numerical target information contained in the text, while preserving semantic descriptions.

In [9]:
df = pd.read_csv(CSV_PATH)

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"
    df[clean_col] = df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print(
        "Remaining numeric leakage:",
        df[clean_col].str.contains(
            r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
            regex=True,
            case=False
        ).sum()
    )

hybrid_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_775/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


text_qwen3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_775/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


vision_gemma3_4b_clean
Remaining numeric leakage: 0


/tmp/ipykernel_775/2462767828.py:40: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[clean_col].str.contains(


In [10]:
TEXT_COLS_CLEAN = [
    "hybrid_gemma3_4b_clean",
    "text_qwen3_4b_clean",
    "vision_gemma3_4b_clean"
]

## Stratified Sampling and Data Split

This section prepares the dataset for training by first filtering out rows with missing values in required columns (images, targets, and cleaned captions).

A dominant class is computed for each sample based on the maximum target value across land cover classes. This is used to perform stratified sampling, ensuring that the class distribution is preserved.

The sampled dataset is split into 80% training, 10% validation, 10% test

Stratification is applied based on the dominant class to maintain consistent class distribution across splits. This ensures fair and reliable evaluation of model performance.

Indices are reset after splitting to avoid indexing issues during data loading.

In [11]:
needed_cols = (
    ["filename"]
    + TARGET_COLS
    + ["hybrid_gemma3_4b_clean", "text_qwen3_4b_clean", "vision_gemma3_4b_clean"]
)

df = df.dropna(subset=needed_cols).copy()

df["dominant_class"] = df[TARGET_COLS].idxmax(axis=1)

# If available rows <= sample_size, use all rows
if len(df) <= CONFIG["sample_size"]:
    sample_df = df.sample(frac=1.0, random_state=CONFIG["seed"]).reset_index(drop=True)
else:
    sample_df, _ = train_test_split(
        df,
        train_size=CONFIG["sample_size"],
        stratify=df["dominant_class"],
        random_state=CONFIG["seed"]
    )
    sample_df = sample_df.reset_index(drop=True)

print("Sample size:", len(sample_df))

train_df, temp_df = train_test_split(
    sample_df,
    test_size=0.20,
    stratify=sample_df["dominant_class"],
    random_state=CONFIG["seed"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["dominant_class"],
    random_state=CONFIG["seed"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Sample size: 10000
Train: 8000
Val: 1000
Test: 1000


## Target Distribution Check

To verify that stratification is effective, this section computes summary statistics (mean and standard deviation) for each target class across the train, validation, and test splits.

This check ensures that the distributions of land cover classes remain consistent across splits, preventing bias in training or evaluation.

In [12]:
def check_target_distribution(train_df, val_df, test_df, target_cols):
    summary = []

    for name, data in [
        ("train", train_df),
        ("val", val_df),
        ("test", test_df)
    ]:
        row = {"split": name, "n": len(data)}

        for col in target_cols:
            row[f"{col}_mean"] = data[col].mean()
            row[f"{col}_std"] = data[col].std()

        summary.append(row)

    return pd.DataFrame(summary)

dist_summary = check_target_distribution(train_df, val_df, test_df, TARGET_COLS)
dist_summary

,split,n,Tree_mean,Tree_std,Shrub_mean,Shrub_std,Grass_mean,Grass_std,Crop_mean,Crop_std,Built-up_mean,Built-up_std,Barren_mean,Barren_std,Water_mean,Water_std
0,train,8000,28.8445,35.170320,0.828875,4.064469,45.367375,33.938682,17.95475,30.544088,1.139625,5.456130,4.192875,11.363640,1.672,9.244490
1,val,1000,28.8610,35.001154,0.818000,3.981297,45.064000,34.437681,18.15700,30.993789,1.190000,4.985053,4.181000,11.631231,1.729,9.926837
2,test,1000,28.1980,34.791741,0.941000,4.581103,45.318000,34.315075,18.14000,30.799334,1.095000,4.951615,4.508000,12.302898,1.800,9.461793


In [13]:
dominant_dist = pd.DataFrame({
    "train": train_df["dominant_class"].value_counts(normalize=True),
    "val": val_df["dominant_class"].value_counts(normalize=True),
    "test": test_df["dominant_class"].value_counts(normalize=True)
}).fillna(0)

dominant_dist

,train,val,test
dominant_class,,,
Grass,0.470375,0.470,0.470
Tree,0.303750,0.304,0.303
Crop,0.180250,0.181,0.180
Barren,0.019250,0.019,0.020
Water,0.017750,0.018,0.018
Built-up,0.005875,0.006,0.006
Shrub,0.002750,0.002,0.003


## Image Transformations

This section defines preprocessing pipelines for training and evaluation.

Training transformations include resizing, random horizontal and vertical flips for data augmentation, and normalization using ImageNet statistics.

Evaluation transformations apply resizing and normalization only, ensuring consistent input without augmentation during validation and testing.

In [14]:
train_tfms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # mean and std values of ImageNet channel
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Multimodal Dataset

Two dataset classes are defined to support different input modalities.

LandCoverDataset handles both image only and image+text inputs. For each sample,
the image is loaded and transformed, and target values are extracted as a
multi-output regression vector. If a tokenizer and text column are provided,
captions are tokenized up to max_token_len tokens. The dataset returns an image
tensor, target vector, and optionally tokenized text inputs (input_ids,
attention_mask).

LandCoverRawTextDataset is used for RemoteCLIP experiments where tokenization
is handled internally by the text encoder. It returns the raw caption string
alongside the image tensor and target vector.

In [15]:
class LandCoverDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, tokenizer=None, text_col=None, max_len=CONFIG["max_token_len"]):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "image": image,
            "target": target
        }

        if self.tokenizer is not None and self.text_col is not None:
            text = str(row[self.text_col])

            enc = self.tokenizer(
                text,
                padding="max_length",
                truncation=True,
                max_length=self.max_len,
                return_tensors="pt"
            )

            item["input_ids"] = enc["input_ids"].squeeze(0)
            item["attention_mask"] = enc["attention_mask"].squeeze(0)

        return item

In [16]:
class LandCoverRawTextDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, text_col=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.text_col = text_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        text = str(row[self.text_col])

        target = torch.tensor(
            row[TARGET_COLS].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image": image,
            "text": text,
            "target": target
        }

## Image Only Baseline Model

This model serves as a baseline using only visual information.

A pretrained vision backbone (DeiT Tiny) is used to extract image features, which
are then passed through a regression head to predict land-cover targets.

The regression head consists of layer normalization, a fully connected hidden layer
of size regressor_hidden_dim, ReLU activation, dropout at rate regressor_dropout,
and a final output layer. Both are configured via CONFIG.

This baseline allows comparison against multimodal models to measure the contribution
of textual information. The backbone is initialized with pretrained weights and
fine-tuned during training.

In [17]:
class ImageOnlyModel(nn.Module):
    def __init__(self, model_name, num_outputs=len(TARGET_COLS)):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.backbone.num_features

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def forward(self, image):
        image_features = self.backbone(image)
        return self.regressor(image_features)

## Multimodal Model (Image + Text Fusion)

This model extends the image-only baseline by incorporating textual information
through a gated fusion mechanism.

The architecture consists of:
- A pretrained vision backbone (DeiT Tiny) for image feature extraction
- A pretrained text encoder (MiniLM / DistilBERT / RemoteCLIP) for caption representation
- A projection layer to map text features into the image feature space

MiniLM and DistilBERT are general-purpose transformer encoders that require
explicit tokenization. ImageTextGatedFrozenScaledModel handles these via
input_ids and attention_mask passed from the DataLoader. RemoteCLIP, by
contrast, is a CLIP-based model pretrained specifically on remote sensing imagery.
It handles tokenization internally, so it uses a separate model class
(DeitTinyRemoteCLIPFusion) that accepts raw text strings directly.

The text encoder is kept frozen during training to reduce computational cost and
ensure stable representations.

To combine modalities, a gating mechanism is used. Image and text features are
concatenated and passed through a learnable gate. The gate controls how much
textual information contributes to the final representation. A scaling factor
(text_scale) further adjusts the strength of text features, and is configured
via CONFIG.

The final fused representation is computed as:

    fused = image_features + scale × gate × text_features

The regression head hidden size and dropout rate are configured via
regressor_hidden_dim and regressor_dropout in CONFIG.

This design allows the model to adaptively use textual information depending on
its relevance.

In [18]:
class ImageTextGatedFrozenScaledModel(nn.Module):
    def __init__(
        self,
        image_model_name,
        text_model_name,
        num_outputs=len(TARGET_COLS),
        text_scale=CONFIG["text_scale"]
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_backbone = AutoModel.from_pretrained(text_model_name)

        for p in self.text_backbone.parameters():
            p.requires_grad = False

        text_dim = self.text_backbone.config.hidden_size

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_backbone(image)

        with torch.no_grad():
            text_output = self.text_backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        text_features = self.mean_pooling(text_output, attention_mask)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused_features = image_features + self.text_scale * gate * text_features

        return self.regressor(fused_features)

In [19]:
class RemoteCLIPTextEncoder(nn.Module):
    def __init__(self, model_name="ViT-B-32"):
        super().__init__()

        self.model, _, _ = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None
        )

        self.tokenizer = open_clip.get_tokenizer(model_name)

        ckpt_path = hf_hub_download(
            repo_id="chendelong/RemoteCLIP",
            filename=f"RemoteCLIP-{model_name}.pt"
        )

        ckpt = torch.load(ckpt_path, map_location="cpu")

        if "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]

        cleaned = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "").replace("model.", "")
            cleaned[k] = v

        self.model.load_state_dict(cleaned, strict=False)

        print("RemoteCLIP weights loaded")

        for p in self.model.parameters():
            p.requires_grad = False

        self.model.eval()

    def forward(self, texts):
        device = next(self.model.parameters()).device
        tokens = self.tokenizer(list(texts)).to(device)

        with torch.no_grad():
            features = self.model.encode_text(tokens)
            features = features / features.norm(dim=-1, keepdim=True)

        return features

In [20]:
class DeitTinyRemoteCLIPFusion(nn.Module):
    def __init__(
        self,
        image_model_name,
        num_outputs=len(TARGET_COLS),
        text_scale=CONFIG["text_scale"]
    ):
        super().__init__()

        self.text_scale = text_scale

        self.image_backbone = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0
        )

        image_dim = self.image_backbone.num_features

        self.text_encoder = RemoteCLIPTextEncoder(model_name="ViT-B-32")

        with torch.no_grad():
            dummy_text = ["satellite image of land cover"]
            dummy_features = self.text_encoder(dummy_text)
            text_dim = dummy_features.shape[1]

        self.text_proj = nn.Linear(text_dim, image_dim)

        self.gate_layer = nn.Sequential(
            nn.Linear(image_dim + image_dim, image_dim),
            nn.Sigmoid()
        )

        hidden = CONFIG["regressor_hidden_dim"]
        self.regressor = nn.Sequential(
            nn.LayerNorm(image_dim),
            nn.Linear(image_dim, hidden),
            nn.ReLU(),
            nn.Dropout(CONFIG["regressor_dropout"]),
            nn.Linear(hidden, num_outputs)
        )

    def forward(self, image, texts):
        image_features = self.image_backbone(image)

        text_features = self.text_encoder(texts)
        text_features = text_features.to(image_features.device)
        text_features = self.text_proj(text_features)

        gate_input = torch.cat([image_features, text_features], dim=1)
        gate = self.gate_layer(gate_input)

        fused = image_features + self.text_scale * gate * text_features

        return self.regressor(fused)

## Evaluation Function

This section defines a unified evaluation function for all model types.

A model_type parameter ("image", "text`, or "remoteclip") controls how
the batch is unpacked and passed to the model, allowing a single function to
handle all three experiment types consistently.

Mean Absolute Error (MAE) is used as the primary evaluation metric. Root Mean
Squared Error (RMSE) is also computed for additional analysis. Class-wise MAE
is calculated to analyze performance across individual land cover categories.

These metrics provide both overall and class level performance insights.

In [21]:
@torch.no_grad()
def evaluate_model(model, loader, config, model_type):
    """
    Unified evaluation for all model types.
    model_type: "image" | "text" | "remoteclip"
    """
    model.eval()

    criterion = nn.L1Loss()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:
        images  = batch["image"].to(config["device"])
        targets = batch["target"].to(config["device"])

        if model_type == "image":
            preds = model(images)
        elif model_type == "text":
            input_ids      = batch["input_ids"].to(config["device"])
            attention_mask = batch["attention_mask"].to(config["device"])
            preds = model(images, input_ids, attention_mask)
        elif model_type == "remoteclip":
            preds = model(images, batch["text"])

        loss = criterion(preds, targets)
        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_targets)

    mae       = mean_absolute_error(y_true, y_pred)
    rmse      = mean_squared_error(y_true, y_pred) ** 0.5
    class_mae = np.mean(np.abs(y_true - y_pred), axis=0)

    return total_loss / len(loader.dataset), mae, rmse, class_mae, y_pred, y_true

## Training Function

This section defines a unified training loop for all model types.

A model_type parameter ("image", "text", or "remoteclip") controls how
the batch is unpacked and passed to the model.
The loss function is Mean Absolute Error (L1 loss). The AdamW optimizer with
weight decay is used. Gradient clipping is applied at each step. Early stopping
is triggered when validation MAE does not improve for early_stopping_patience
consecutive epochs, at which point the best model state is restored.

Training progress is logged to Weights & Biases (W&B) each epoch.

In [22]:
def train_model(model, train_loader, val_loader, config, model_type):
    """
    Unified training loop for all model types.
    model_type: "image" | "text" | "remoteclip"
    """
    criterion = nn.L1Loss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    best_val_mae    = float("inf")
    best_state      = None
    patience_counter = 0
    history         = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            images  = batch["image"].to(config["device"])
            targets = batch["target"].to(config["device"])

            if model_type == "image":
                preds = model(images)
            elif model_type == "text":
                input_ids      = batch["input_ids"].to(config["device"])
                attention_mask = batch["attention_mask"].to(config["device"])
                preds = model(images, input_ids, attention_mask)
            elif model_type == "remoteclip":
                preds = model(images, batch["text"])

            loss = criterion(preds, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip"])
            optimizer.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)

        val_loss, val_mae, val_rmse, _, _, _ = evaluate_model(
            model, val_loader, config, model_type
        )

        history.append({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_mae":    val_mae,
            "val_rmse":   val_rmse
        })

        print(
            f"Epoch {epoch+1}/{config['epochs']} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        wandb.log({
            "epoch":      epoch + 1,
            "train_loss": train_loss,
            "val_loss":   val_loss,
            "val_mae":    val_mae,
            "val_rmse":   val_rmse
        })

        if val_mae < best_val_mae:
            best_val_mae     = val_mae
            best_state       = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= config["early_stopping_patience"]:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_val_mae

## Experiment: Image Only Baseline

This function runs the image only experiment using the DeiT Tiny backbone.

The full CONFIG is logged to W&B as the run configuration. The wandb_project
parameter defaults to CONFIG["wandb_project"] to avoid hardcoding. If
save_predictions is set to True, test set predictions and ground truth values
are saved as a CSV to CONFIG["predictions_dir"].

Metrics (MAE, RMSE, and class-wise MAE) are logged to Weights & Biases (W&B).

This experiment serves as the baseline for comparing multimodal configurations.

In [23]:
def run_deittiny_image_only(
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":   "deittiny_image_only",
        "fusion":       "none",
        "text_column":  "none",
        "text_model":   "none",
    }

    wandb.init(
        project=wandb_project,
        name="deittiny_image_only",
        config=run_config,
        reinit=True
    )

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = ImageOnlyModel(
        model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS)
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="image")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="image"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_image_only.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment": "deittiny_image_only",
        "text_model": "none",
        "test_mae":   test_mae,
        "test_rmse":  test_rmse,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

## Experiment: Multimodal (Image + Text)

This function runs multimodal experiments by combining image features with textual
information using transformer-based text encoders (MiniLM, DistilBERT).

MiniLM and DistilBERT are lightweight transformer encoders that process tokenized
text. They require explicit tokenization before the forward pass, input IDs and
attention masks are prepared by the DataLoader via LandCoverDataset and passed
directly to the model. Therefore run_deittiny_transformer_text uses
model_type="text" and LandCoverDataset with a tokenizer.

RemoteCLIP, by contrast, handles its own tokenization internally inside the text
encoder's forward pass and expects raw caption strings. This is why it uses a
separate run_deittiny_remoteclip_text function with model_type="remoteclip"
and LandCoverRawTextDataset instead.

The text_col and text_model parameters specify the caption source and encoder
for each run. text_scale defaults to CONFIG["text_scale"] and wandb_project
defaults to CONFIG["wandb_project"] to avoid hardcoding. If save_predictions
is set to True, test set predictions are saved to CONFIG["predictions_dir"].

Results are logged to W&B, enabling comparison across different caption sources
and text encoders.

In [24]:
def run_deittiny_transformer_text(
    text_col,
    text_model,
    text_scale=CONFIG["text_scale"],
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":   "deittiny_transformer_text",
        "fusion":       "gated_frozen_scaled",
        "text_column":  text_col,
        "text_model":   text_model,
        "text_scale":   text_scale,
    }

    run_name = f"deittiny_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}"

    wandb.init(project=wandb_project, name=run_name, config=run_config, reinit=True)

    tokenizer = AutoTokenizer.from_pretrained(text_model)

    train_ds = LandCoverDataset(train_df, IMAGE_DIR, transform=train_tfms, tokenizer=tokenizer, text_col=text_col)
    val_ds   = LandCoverDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)
    test_ds  = LandCoverDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  tokenizer=tokenizer, text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = ImageTextGatedFrozenScaledModel(
        image_model_name=CONFIG["model_name"],
        text_model_name=text_model,
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="text")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="text"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_{text_col}_{text_model.split('/')[-1]}_scale_{text_scale}.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse, "best_val_mae": best_val_mae})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  text_model,
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

In [25]:
def run_deittiny_remoteclip_text(
    text_col,
    text_scale=CONFIG["text_scale"],
    wandb_project=CONFIG["wandb_project"],
    save_predictions=False
):
    seed_everything(CONFIG["seed"])

    run_config = {
        **CONFIG,
        "experiment":  "deittiny_remoteclip_text",
        "fusion":      "gated_frozen_scaled",
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
    }

    run_name = f"deittiny_remoteclip_{text_col}_scale_{text_scale}"

    wandb.init(project=wandb_project, name=run_name, config=run_config, reinit=True)

    train_ds = LandCoverRawTextDataset(train_df, IMAGE_DIR, transform=train_tfms, text_col=text_col)
    val_ds   = LandCoverRawTextDataset(val_df,   IMAGE_DIR, transform=eval_tfms,  text_col=text_col)
    test_ds  = LandCoverRawTextDataset(test_df,  IMAGE_DIR, transform=eval_tfms,  text_col=text_col)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

    model = DeitTinyRemoteCLIPFusion(
        image_model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS),
        text_scale=text_scale
    ).to(CONFIG["device"])

    model, history, best_val_mae = train_model(model, train_loader, val_loader, CONFIG, model_type="remoteclip")

    test_loss, test_mae, test_rmse, class_mae, y_pred, y_true = evaluate_model(
        model, test_loader, CONFIG, model_type="remoteclip"
    )

    if save_predictions:
        pred_path = os.path.join(
            CONFIG["predictions_dir"],
            f"predictions_{CONFIG['model_name']}_{text_col}_remoteclip_scale_{text_scale}.csv"
        )
        pd.DataFrame({
            "filename": test_df["filename"].values,
            **{f"pred_{cls}": y_pred[:, i] for i, cls in enumerate(TARGET_COLS)},
            **{f"true_{cls}": y_true[:, i] for i, cls in enumerate(TARGET_COLS)}
        }).to_csv(pred_path, index=False)

    wandb.log({"test_loss": test_loss, "test_mae": test_mae, "test_rmse": test_rmse, "best_val_mae": best_val_mae})
    wandb.log({
        "class_mae_table": wandb.Table(
            dataframe=pd.DataFrame({"class": TARGET_COLS, "class_mae": class_mae})
        )
    })
    wandb.finish()

    return {
        "experiment":  run_name,
        "text_column": text_col,
        "text_model":  "RemoteCLIP ViT-B-32",
        "text_scale":  text_scale,
        "val_mae":     best_val_mae,
        "test_mae":    test_mae,
        "test_rmse":   test_rmse,
        "class_mae":   class_mae,
        **{f"mae_{cls}": class_mae[i] for i, cls in enumerate(TARGET_COLS)}
    }

# Running Experiments

In [26]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ayseatci00 (ayseatci00-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [27]:
results = []

## Image Only Experiment

In [28]:
results.append(run_deittiny_image_only(save_predictions=True))
pd.DataFrame(results)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

Epoch 1/15 | Train Loss: 12.6414 | Val MAE: 11.1571 | Val RMSE: 25.1707
Epoch 2/15 | Train Loss: 9.7581 | Val MAE: 7.9412 | Val RMSE: 18.3232
Epoch 3/15 | Train Loss: 7.1854 | Val MAE: 5.7686 | Val RMSE: 13.5642
Epoch 4/15 | Train Loss: 5.4667 | Val MAE: 4.1626 | Val RMSE: 10.5868
Epoch 5/15 | Train Loss: 4.4775 | Val MAE: 3.8366 | Val RMSE: 10.1583
Epoch 6/15 | Train Loss: 3.9982 | Val MAE: 3.4500 | Val RMSE: 8.8247
Epoch 7/15 | Train Loss: 3.6941 | Val MAE: 3.3676 | Val RMSE: 8.9763
Epoch 8/15 | Train Loss: 3.4955 | Val MAE: 3.3752 | Val RMSE: 9.4705
Epoch 9/15 | Train Loss: 3.3891 | Val MAE: 3.1896 | Val RMSE: 8.2457
Epoch 10/15 | Train Loss: 3.2324 | Val MAE: 2.7505 | Val RMSE: 7.6799
Epoch 11/15 | Train Loss: 3.1301 | Val MAE: 2.9491 | Val RMSE: 8.0948
Epoch 12/15 | Train Loss: 3.0206 | Val MAE: 2.6964 | Val RMSE: 7.5676
Epoch 13/15 | Train Loss: 2.9039 | Val MAE: 2.7345 | Val RMSE: 7.6481
Epoch 14/15 | Train Loss: 2.7911 | Val MAE: 2.7159 | Val RMSE: 7.3122
Epoch 15/15 | Train Lo

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▅▄▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▅▄▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▅▃▂▂▂▂▂▁▁▁▁▁▁▁
epoch,15
test_loss,2.81209
test_mae,2.81209


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,deittiny_image_only,none,2.812086,7.65936,2.827943,0.965573,6.366722,5.030444,0.871207,2.602988,1.019728


## Multimodal Experiments
Multimodal experiments are conducted using three caption types (text only, hybrid, and vision based) and three text encoders (MiniLM, RemoteCLIP and DistilBERT). A fixed text scaling factor is applied to control the contribution of textual features in the fusion process.

### MiniLM Experiments

In [29]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_deittiny_transformer_text(
            text_col=text_col,
            text_model="sentence-transformers/all-MiniLM-L6-v2",
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▄▂▂▂▁▁▁▁▁▁▁▁▁
val_mae,█▆▄▂▂▂▁▁▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
best_val_mae,2.37665
epoch,15


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Epoch 1/15 | Train Loss: 12.7382 | Val MAE: 11.2764 | Val RMSE: 25.1388
Epoch 2/15 | Train Loss: 10.0352 | Val MAE: 8.6515 | Val RMSE: 19.2041
Epoch 3/15 | Train Loss: 7.4962 | Val MAE: 5.7839 | Val RMSE: 13.8124
Epoch 4/15 | Train Loss: 5.6446 | Val MAE: 4.4227 | Val RMSE: 10.8898
Epoch 5/15 | Train Loss: 4.6187 | Val MAE: 3.7287 | Val RMSE: 9.2164
Epoch 6/15 | Train Loss: 4.0734 | Val MAE: 3.6029 | Val RMSE: 8.7075
Epoch 7/15 | Train Loss: 3.7885 | Val MAE: 3.1749 | Val RMSE: 8.4146
Epoch 8/15 | Train Loss: 3.6125 | Val MAE: 2.9283 | Val RMSE: 7.9556
Epoch 9/15 | Train Loss: 3.4051 | Val MAE: 3.2953 | Val RMSE: 8.6197
Epoch 10/15 | Train Loss: 3.2728 | Val MAE: 3.1079 | Val RMSE: 8.0893
Epoch 11/15 | Train Loss: 3.1051 | Val MAE: 2.9963 | Val RMSE: 8.0463
Early stopping triggered.


best_val_mae,▁
epoch,▁▂▂▃▄▅▅▆▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▁▁▁▁▁
val_loss,█▆▃▂▂▂▁▁▁▁▁
val_mae,█▆▃▂▂▂▁▁▁▁▁
val_rmse,█▆▃▂▂▁▁▁▁▁▁
best_val_mae,2.92829
epoch,11


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,sentence-transformers/all-MiniLM-L6-v2,2.397682,6.197832,2.794325,0.949530,5.257837,3.386382,0.745457,2.552549,1.097693,text_qwen3_4b_clean,0.7,2.376651,"[2.794325, 0.9495305, 5.2578373, 3.3863816, 0...."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.518511,6.544519,3.021349,0.953259,5.379640,3.826561,0.805188,2.529843,1.113733,hybrid_gemma3_4b_clean,0.7,2.481461,"[3.0213492, 0.95325917, 5.3796396, 3.826561, 0..."
0,deittiny_image_only,none,2.812086,7.659360,2.827943,0.965573,6.366722,5.030444,0.871207,2.602988,1.019728,NaN,NaN,NaN,NaN
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,3.090967,8.295122,2.892072,0.966379,6.978871,5.252649,1.050385,2.896824,1.599588,vision_gemma3_4b_clean,0.7,2.928293,"[2.8920717, 0.9663787, 6.978871, 5.2526493, 1...."


### Remote CLIP Experiments

In [30]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_deittiny_remoteclip_text(
            text_col=text_col,
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 12.7802 | Val MAE: 11.3445 | Val RMSE: 25.4372
Epoch 2/15 | Train Loss: 10.1224 | Val MAE: 8.4382 | Val RMSE: 19.3334
Epoch 3/15 | Train Loss: 7.5243 | Val MAE: 5.8441 | Val RMSE: 13.8380
Epoch 4/15 | Train Loss: 5.5828 | Val MAE: 4.3181 | Val RMSE: 10.7852
Epoch 5/15 | Train Loss: 4.5888 | Val MAE: 3.8103 | Val RMSE: 9.4656
Epoch 6/15 | Train Loss: 4.0121 | Val MAE: 3.0873 | Val RMSE: 8.3172
Epoch 7/15 | Train Loss: 3.7201 | Val MAE: 3.0307 | Val RMSE: 8.2339
Epoch 8/15 | Train Loss: 3.5109 | Val MAE: 3.0573 | Val RMSE: 8.0326
Epoch 9/15 | Train Loss: 3.3477 | Val MAE: 2.8344 | Val RMSE: 7.8773
Epoch 10/15 | Train Loss: 3.2517 | Val MAE: 2.9139 | Val RMSE: 7.8010
Epoch 11/15 | Train Loss: 3.0986 | Val MAE: 2.6221 | Val RMSE: 7.3066
Epoch 12/15 | Train Loss: 2.9533 | Val MAE: 2.7591 | Val RMSE: 7.3566
Epoch 13/15 | Train Loss: 2.8212 | Val MAE: 2.6735 | Val RMSE: 7.3124
Epoch 14/15 | Train Loss: 2.6837 | Val MAE: 2.4991 | Val RMSE: 6.8

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▂▂▁▁▁▁▁▁▁▁▁▁
val_mae,█▆▄▂▂▁▁▁▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
best_val_mae,2.49908
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 12.7738 | Val MAE: 11.4261 | Val RMSE: 25.5170
Epoch 2/15 | Train Loss: 10.1333 | Val MAE: 8.4154 | Val RMSE: 19.3014
Epoch 3/15 | Train Loss: 7.4304 | Val MAE: 5.9126 | Val RMSE: 13.9288
Epoch 4/15 | Train Loss: 5.6637 | Val MAE: 4.3654 | Val RMSE: 10.7885
Epoch 5/15 | Train Loss: 4.6308 | Val MAE: 3.6119 | Val RMSE: 9.1232
Epoch 6/15 | Train Loss: 3.9893 | Val MAE: 3.4645 | Val RMSE: 8.7747
Epoch 7/15 | Train Loss: 3.6509 | Val MAE: 3.1004 | Val RMSE: 8.2950
Epoch 8/15 | Train Loss: 3.4199 | Val MAE: 2.8680 | Val RMSE: 7.7148
Epoch 9/15 | Train Loss: 3.2494 | Val MAE: 2.8092 | Val RMSE: 7.4859
Epoch 10/15 | Train Loss: 3.0619 | Val MAE: 2.6087 | Val RMSE: 7.0695
Epoch 11/15 | Train Loss: 2.9359 | Val MAE: 2.5036 | Val RMSE: 6.6847
Epoch 12/15 | Train Loss: 2.8180 | Val MAE: 2.6236 | Val RMSE: 7.0461
Epoch 13/15 | Train Loss: 2.7115 | Val MAE: 2.4655 | Val RMSE: 6.4852
Epoch 14/15 | Train Loss: 2.6004 | Val MAE: 2.4742 | Val RMSE: 6.5

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
best_val_mae,2.31358
epoch,15


RemoteCLIP weights loaded
Epoch 1/15 | Train Loss: 12.7700 | Val MAE: 11.3277 | Val RMSE: 25.5087
Epoch 2/15 | Train Loss: 10.1059 | Val MAE: 8.4140 | Val RMSE: 19.1178
Epoch 3/15 | Train Loss: 7.4049 | Val MAE: 5.9656 | Val RMSE: 13.8850
Epoch 4/15 | Train Loss: 5.6573 | Val MAE: 4.5904 | Val RMSE: 11.1325
Epoch 5/15 | Train Loss: 4.7019 | Val MAE: 3.7195 | Val RMSE: 9.4870
Epoch 6/15 | Train Loss: 4.1635 | Val MAE: 3.3940 | Val RMSE: 8.7051
Epoch 7/15 | Train Loss: 3.8133 | Val MAE: 3.0764 | Val RMSE: 8.1942
Epoch 8/15 | Train Loss: 3.5683 | Val MAE: 3.1264 | Val RMSE: 8.2475
Epoch 9/15 | Train Loss: 3.4210 | Val MAE: 2.9892 | Val RMSE: 7.9000
Epoch 10/15 | Train Loss: 3.2647 | Val MAE: 2.8971 | Val RMSE: 7.9369
Epoch 11/15 | Train Loss: 3.1427 | Val MAE: 2.7975 | Val RMSE: 7.7095
Epoch 12/15 | Train Loss: 3.0238 | Val MAE: 3.1199 | Val RMSE: 7.9764
Epoch 13/15 | Train Loss: 2.9529 | Val MAE: 2.8418 | Val RMSE: 7.6594
Epoch 14/15 | Train Loss: 2.7520 | Val MAE: 2.7265 | Val RMSE: 7.3

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁
val_rmse,█▆▄▂▂▂▁▁▁▁▁▁▁▁▁
best_val_mae,2.7265
epoch,15


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
5,deittiny_remoteclip_text_qwen3_4b_clean_scale_0.7,RemoteCLIP ViT-B-32,2.379557,6.205260,2.554680,0.950755,5.274544,3.661830,1.056511,2.420975,0.737603,text_qwen3_4b_clean,0.7,2.313581,"[2.55468, 0.9507553, 5.2745442, 3.66183, 1.056..."
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,sentence-transformers/all-MiniLM-L6-v2,2.397682,6.197832,2.794325,0.949530,5.257837,3.386382,0.745457,2.552549,1.097693,text_qwen3_4b_clean,0.7,2.376651,"[2.794325, 0.9495305, 5.2578373, 3.3863816, 0...."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.518511,6.544519,3.021349,0.953259,5.379640,3.826561,0.805188,2.529843,1.113733,hybrid_gemma3_4b_clean,0.7,2.481461,"[3.0213492, 0.95325917, 5.3796396, 3.826561, 0..."
4,deittiny_remoteclip_hybrid_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.697665,6.951913,3.072670,0.961231,6.269224,4.102949,0.944405,2.612420,0.920758,hybrid_gemma3_4b_clean,0.7,2.499080,"[3.07267, 0.9612306, 6.2692237, 4.1029487, 0.9..."
0,deittiny_image_only,none,2.812086,7.659360,2.827943,0.965573,6.366722,5.030444,0.871207,2.602988,1.019728,NaN,NaN,NaN,NaN
6,deittiny_remoteclip_vision_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.855073,7.644063,2.888341,0.955762,6.513663,5.062417,0.995570,2.618411,0.951351,vision_gemma3_4b_clean,0.7,2.726500,"[2.8883414, 0.9557616, 6.513663, 5.062417, 0.9..."
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,3.090967,8.295122,2.892072,0.966379,6.978871,5.252649,1.050385,2.896824,1.599588,vision_gemma3_4b_clean,0.7,2.928293,"[2.8920717, 0.9663787, 6.978871, 5.2526493, 1...."


### DistilBERT Experiments

In [31]:
for text_col in TEXT_COLS_CLEAN:
    results.append(
        run_deittiny_transformer_text(
            text_col=text_col,
            text_model="distilbert-base-uncased",
            text_scale=CONFIG["text_scale"],
            save_predictions=True
        )
    )

pd.DataFrame(results).sort_values("test_mae")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.8847 | Val MAE: 11.4483 | Val RMSE: 25.7416
Epoch 2/15 | Train Loss: 10.2839 | Val MAE: 8.6231 | Val RMSE: 19.3684
Epoch 3/15 | Train Loss: 7.6132 | Val MAE: 5.9552 | Val RMSE: 13.9245
Epoch 4/15 | Train Loss: 5.7370 | Val MAE: 4.7900 | Val RMSE: 11.2786
Epoch 5/15 | Train Loss: 4.6641 | Val MAE: 3.6979 | Val RMSE: 9.3765
Epoch 6/15 | Train Loss: 4.1185 | Val MAE: 3.4942 | Val RMSE: 8.8778
Epoch 7/15 | Train Loss: 3.8086 | Val MAE: 3.3148 | Val RMSE: 8.8472
Epoch 8/15 | Train Loss: 3.6233 | Val MAE: 3.1013 | Val RMSE: 8.3642
Epoch 9/15 | Train Loss: 3.4167 | Val MAE: 2.9753 | Val RMSE: 8.2970
Epoch 10/15 | Train Loss: 3.3214 | Val MAE: 2.9892 | Val RMSE: 8.2529
Epoch 11/15 | Train Loss: 3.1528 | Val MAE: 3.0009 | Val RMSE: 8.1018
Epoch 12/15 | Train Loss: 3.0264 | Val MAE: 3.0412 | Val RMSE: 8.0212
Early stopping triggered.


best_val_mae,▁
epoch,▁▂▂▃▄▄▅▅▆▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▁▁▁▁▁
val_loss,█▆▃▂▂▁▁▁▁▁▁▁
val_mae,█▆▃▂▂▁▁▁▁▁▁▁
val_rmse,█▅▃▂▂▁▁▁▁▁▁▁
best_val_mae,2.97528
epoch,12


,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
8,deittiny_text_qwen3_4b_clean_distilbert-base-u...,distilbert-base-uncased,2.323391,6.055197,2.535043,0.954727,5.098788,3.471630,0.721334,2.484310,0.997902,text_qwen3_4b_clean,0.7,2.285313,"[2.5350432, 0.9547271, 5.0987883, 3.4716299, 0..."
5,deittiny_remoteclip_text_qwen3_4b_clean_scale_0.7,RemoteCLIP ViT-B-32,2.379557,6.205260,2.554680,0.950755,5.274544,3.661830,1.056511,2.420975,0.737603,text_qwen3_4b_clean,0.7,2.313581,"[2.55468, 0.9507553, 5.2745442, 3.66183, 1.056..."
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,sentence-transformers/all-MiniLM-L6-v2,2.397682,6.197832,2.794325,0.949530,5.257837,3.386382,0.745457,2.552549,1.097693,text_qwen3_4b_clean,0.7,2.376651,"[2.794325, 0.9495305, 5.2578373, 3.3863816, 0...."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.518511,6.544519,3.021349,0.953259,5.379640,3.826561,0.805188,2.529843,1.113733,hybrid_gemma3_4b_clean,0.7,2.481461,"[3.0213492, 0.95325917, 5.3796396, 3.826561, 0..."
7,deittiny_hybrid_gemma3_4b_clean_distilbert-bas...,distilbert-base-uncased,2.591328,6.873420,2.895484,0.954130,5.882454,4.325740,0.739391,2.418660,0.923430,hybrid_gemma3_4b_clean,0.7,2.482271,"[2.8954842, 0.9541302, 5.8824544, 4.3257403, 0..."
4,deittiny_remoteclip_hybrid_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.697665,6.951913,3.072670,0.961231,6.269224,4.102949,0.944405,2.612420,0.920758,hybrid_gemma3_4b_clean,0.7,2.499080,"[3.07267, 0.9612306, 6.2692237, 4.1029487, 0.9..."
0,deittiny_image_only,none,2.812086,7.659360,2.827943,0.965573,6.366722,5.030444,0.871207,2.602988,1.019728,NaN,NaN,NaN,NaN
6,deittiny_remoteclip_vision_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.855073,7.644063,2.888341,0.955762,6.513663,5.062417,0.995570,2.618411,0.951351,vision_gemma3_4b_clean,0.7,2.726500,"[2.8883414, 0.9557616, 6.513663, 5.062417, 0.9..."
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,3.090967,8.295122,2.892072,0.966379,6.978871,5.252649,1.050385,2.896824,1.599588,vision_gemma3_4b_clean,0.7,2.928293,"[2.8920717, 0.9663787, 6.978871, 5.2526493, 1...."
9,deittiny_vision_gemma3_4b_clean_distilbert-bas...,distilbert-base-uncased,3.101354,7.963865,3.481380,0.969705,7.091200,4.946319,0.952847,2.979886,1.288141,vision_gemma3_4b_clean,0.7,2.975283,"[3.4813795, 0.96970534, 7.0912, 4.946319, 0.95..."


### Deit Tiny Experiment Results Summary

In [32]:
results_df = pd.DataFrame(results).sort_values("test_mae")
results_df

,experiment,text_model,test_mae,test_rmse,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water,text_column,text_scale,val_mae,class_mae
8,deittiny_text_qwen3_4b_clean_distilbert-base-u...,distilbert-base-uncased,2.323391,6.055197,2.535043,0.954727,5.098788,3.471630,0.721334,2.484310,0.997902,text_qwen3_4b_clean,0.7,2.285313,"[2.5350432, 0.9547271, 5.0987883, 3.4716299, 0..."
5,deittiny_remoteclip_text_qwen3_4b_clean_scale_0.7,RemoteCLIP ViT-B-32,2.379557,6.205260,2.554680,0.950755,5.274544,3.661830,1.056511,2.420975,0.737603,text_qwen3_4b_clean,0.7,2.313581,"[2.55468, 0.9507553, 5.2745442, 3.66183, 1.056..."
2,deittiny_text_qwen3_4b_clean_all-MiniLM-L6-v2_...,sentence-transformers/all-MiniLM-L6-v2,2.397682,6.197832,2.794325,0.949530,5.257837,3.386382,0.745457,2.552549,1.097693,text_qwen3_4b_clean,0.7,2.376651,"[2.794325, 0.9495305, 5.2578373, 3.3863816, 0...."
1,deittiny_hybrid_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,2.518511,6.544519,3.021349,0.953259,5.379640,3.826561,0.805188,2.529843,1.113733,hybrid_gemma3_4b_clean,0.7,2.481461,"[3.0213492, 0.95325917, 5.3796396, 3.826561, 0..."
7,deittiny_hybrid_gemma3_4b_clean_distilbert-bas...,distilbert-base-uncased,2.591328,6.873420,2.895484,0.954130,5.882454,4.325740,0.739391,2.418660,0.923430,hybrid_gemma3_4b_clean,0.7,2.482271,"[2.8954842, 0.9541302, 5.8824544, 4.3257403, 0..."
4,deittiny_remoteclip_hybrid_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.697665,6.951913,3.072670,0.961231,6.269224,4.102949,0.944405,2.612420,0.920758,hybrid_gemma3_4b_clean,0.7,2.499080,"[3.07267, 0.9612306, 6.2692237, 4.1029487, 0.9..."
0,deittiny_image_only,none,2.812086,7.659360,2.827943,0.965573,6.366722,5.030444,0.871207,2.602988,1.019728,NaN,NaN,NaN,NaN
6,deittiny_remoteclip_vision_gemma3_4b_clean_sca...,RemoteCLIP ViT-B-32,2.855073,7.644063,2.888341,0.955762,6.513663,5.062417,0.995570,2.618411,0.951351,vision_gemma3_4b_clean,0.7,2.726500,"[2.8883414, 0.9557616, 6.513663, 5.062417, 0.9..."
3,deittiny_vision_gemma3_4b_clean_all-MiniLM-L6-...,sentence-transformers/all-MiniLM-L6-v2,3.090967,8.295122,2.892072,0.966379,6.978871,5.252649,1.050385,2.896824,1.599588,vision_gemma3_4b_clean,0.7,2.928293,"[2.8920717, 0.9663787, 6.978871, 5.2526493, 1...."
9,deittiny_vision_gemma3_4b_clean_distilbert-bas...,distilbert-base-uncased,3.101354,7.963865,3.481380,0.969705,7.091200,4.946319,0.952847,2.979886,1.288141,vision_gemma3_4b_clean,0.7,2.975283,"[3.4813795, 0.96970534, 7.0912, 4.946319, 0.95..."


Experiments are conducted using three caption types (text only, hybrid, and
vision based) and three text encoders (MiniLM, RemoteCLIP, and DistilBERT).
A fixed text scaling factor of 0.7 is applied to control the contribution of
textual features in the fusion process.

The image only baseline achieves a test MAE of 2.812. Incorporating textual
information leads to improvements for text only and hybrid captions, while
vision based captions fail to improve over the baseline.

The best performance is obtained with text only captions and DistilBERT (MAE:
2.323), followed by RemoteCLIP on text only captions (MAE: 2.380) and MiniLM
(MAE: 2.398). Notably, RemoteCLIP ranks second overall for DeiT Tiny, performing
more competitively here than in SwinV2 and MaxViT experiments.

Hybrid captions provide moderate improvements, with MiniLM (MAE: 2.519) and
DistilBERT (MAE: 2.591) outperforming the baseline, while RemoteCLIP on hybrid
captions (MAE: 2.698) only marginally improves over it.

Vision based captions perform at or below the baseline across all encoders, with
MAE ranging from 2.855 to 3.101, confirming that visually derived captions do
not provide useful complementary information for DeiT Tiny.

Class-wise results show the largest errors on Grass and Crop across all
configurations. Text only captions with DistilBERT provide the most notable
reductions in these classes compared to the baseline, while Shrub remains
consistently low and largely unaffected across all configurations.

Overall, DeiT Tiny shows a larger gap between the best multimodal configuration
and the baseline compared to SwinV2 and MaxViT, suggesting the weaker visual
backbone benefits more from high quality textual information.

## Text Scale Experiments

The best caption × encoder combination for DeiT Tiny based on
validation MAE at scale 0.7 is selected.

In [33]:
# Filter out image-only row — selection is over multimodal configs only
multimodal_results = [r for r in results if r.get("text_model") not in [None, "none"]]
results_df_multimodal = pd.DataFrame(multimodal_results)

best_config = results_df_multimodal.loc[results_df_multimodal["val_mae"].idxmin()]

print("Best configuration for DeiT-Tiny (by validation MAE)")
print(f"  Text column : {best_config['text_column']}")
print(f"  Text model  : {best_config['text_model']}")
print(f"  Val MAE     : {best_config['val_mae']:.4f}")
print(f"  Test MAE    : {best_config['test_mae']:.4f}")

BEST_TEXT_COL   = best_config["text_column"]
BEST_TEXT_MODEL = best_config["text_model"]
IS_REMOTECLIP   = (BEST_TEXT_MODEL == "RemoteCLIP ViT-B-32")

Best configuration for DeiT-Tiny (by validation MAE)
  Text column : text_qwen3_4b_clean
  Text model  : distilbert-base-uncased
  Val MAE     : 2.2853
  Test MAE    : 2.3234


### Text Scale Ablation

Using the best configuration selected above, text_scale across
[0.3, 0.5, 0.7, 1.0, 1.5] is sweeped.

In [34]:
TEXT_SCALES = [0.3, 0.5, 0.7, 1.0, 1.5]

scale_results = []

for scale in TEXT_SCALES:
    print(f"\nRunning scale={scale} | col={BEST_TEXT_COL} | model={BEST_TEXT_MODEL}")

    if IS_REMOTECLIP:
        result = run_deittiny_remoteclip_text(
            text_col=BEST_TEXT_COL,
            text_scale=scale,
            wandb_project=CONFIG["wandb_project_scale"],
            save_predictions=True
        )
    else:
        result = run_deittiny_transformer_text(
            text_col=BEST_TEXT_COL,
            text_model=BEST_TEXT_MODEL,
            text_scale=scale,
            wandb_project=CONFIG["wandb_project_scale"],
            save_predictions=True
        )

    scale_results.append(result)

scale_df = pd.DataFrame(scale_results).sort_values("text_scale").reset_index(drop=True)
scale_df


Running scale=0.3 | col=text_qwen3_4b_clean | model=distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.8795 | Val MAE: 11.4603 | Val RMSE: 25.8222
Epoch 2/15 | Train Loss: 10.2285 | Val MAE: 8.4586 | Val RMSE: 19.2925
Epoch 3/15 | Train Loss: 7.4414 | Val MAE: 6.0899 | Val RMSE: 13.9098
Epoch 4/15 | Train Loss: 5.6344 | Val MAE: 4.6105 | Val RMSE: 10.8103
Epoch 5/15 | Train Loss: 4.5424 | Val MAE: 3.5479 | Val RMSE: 9.1575
Epoch 6/15 | Train Loss: 3.9343 | Val MAE: 3.1672 | Val RMSE: 8.4240
Epoch 7/15 | Train Loss: 3.6377 | Val MAE: 3.3482 | Val RMSE: 8.4615
Epoch 8/15 | Train Loss: 3.4422 | Val MAE: 2.9844 | Val RMSE: 7.8440
Epoch 9/15 | Train Loss: 3.2684 | Val MAE: 2.8262 | Val RMSE: 7.6689
Epoch 10/15 | Train Loss: 3.1521 | Val MAE: 2.7177 | Val RMSE: 7.4194
Epoch 11/15 | Train Loss: 2.9843 | Val MAE: 2.5670 | Val RMSE: 7.0117
Epoch 12/15 | Train Loss: 2.8017 | Val MAE: 2.6053 | Val RMSE: 6.9822
Epoch 13/15 | Train Loss: 2.6949 | Val MAE: 2.5162 | Val RMSE: 6.6606
Epoch 14/15 | Train Loss: 2.5997 | Val MAE: 2.4618 | Val RMSE: 6.5145
Epoch 15/15 | Train Lo

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▁▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
val_mae,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▁▁▁▁▁▁▁▁
best_val_mae,2.46182
epoch,15


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/15 | Train Loss: 12.8664 | Val MAE: 11.4550 | Val RMSE: 25.7443
Epoch 2/15 | Train Loss: 10.2269 | Val MAE: 8.6339 | Val RMSE: 19.4293
Epoch 3/15 | Train Loss: 7.4604 | Val MAE: 6.0683 | Val RMSE: 13.8050
Epoch 4/15 | Train Loss: 5.4942 | Val MAE: 4.1174 | Val RMSE: 10.0396
Epoch 5/15 | Train Loss: 4.3448 | Val MAE: 3.4976 | Val RMSE: 8.7223
Epoch 6/15 | Train Loss: 3.7702 | Val MAE: 3.2018 | Val RMSE: 8.2118
Epoch 7/15 | Train Loss: 3.5887 | Val MAE: 2.9780 | Val RMSE: 7.9231
Epoch 8/15 | Train Loss: 3.3398 | Val MAE: 2.8886 | Val RMSE: 7.4729
Epoch 9/15 | Train Loss: 3.2023 | Val MAE: 2.7335 | Val RMSE: 7.4668
Epoch 10/15 | Train Loss: 3.0544 | Val MAE: 2.4963 | Val RMSE: 7.0116
Epoch 11/15 | Train Loss: 2.8766 | Val MAE: 2.5063 | Val RMSE: 6.7452
Epoch 12/15 | Train Loss: 2.7531 | Val MAE: 2.5010 | Val RMSE: 6.7101
Epoch 13/15 | Train Loss: 2.6296 | Val MAE: 2.3685 | Val RMSE: 6.4357
Epoch 14/15 | Train Loss: 2.5130 | Val MAE: 2.2307 | Val RMSE: 5.9808
Epoch 15/15 | Train Lo

best_val_mae,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
test_loss,▁
test_mae,▁
test_rmse,▁
train_loss,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
val_loss,█▆▄▂▂▂▂▂▁▁▁▁▁▁▁
val_mae,█▆▄▂▂▂▂▂▁▁▁▁▁▁▁
val_rmse,█▆▄▃▂▂▂▂▂▁▁▁▁▁▁
best_val_mae,2.1382
epoch,15


,experiment,text_column,text_model,text_scale,val_mae,test_mae,test_rmse,class_mae,mae_Tree,mae_Shrub,mae_Grass,mae_Crop,mae_Built-up,mae_Barren,mae_Water
0,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,0.3,2.461822,2.544542,6.554314,"[2.9158633, 0.9485049, 5.923044, 3.7607746, 0....",2.915863,0.948505,5.923044,3.760775,0.644541,2.752447,0.866620
1,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,0.5,2.226678,2.325318,5.892409,"[2.9046955, 0.95601326, 5.1345215, 3.2640562, ...",2.904696,0.956013,5.134521,3.264056,0.703027,2.607940,0.706970
2,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,0.7,2.237237,2.262457,5.832594,"[2.558938, 0.9541037, 4.9508004, 3.2639294, 0....",2.558938,0.954104,4.950800,3.263929,0.722657,2.481685,0.905083
3,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,1.0,2.236047,2.269548,5.808817,"[2.6956544, 0.9519094, 4.841776, 3.3019114, 0....",2.695654,0.951909,4.841776,3.301911,0.770812,2.544267,0.780507
4,deittiny_text_qwen3_4b_clean_distilbert-base-u...,text_qwen3_4b_clean,distilbert-base-uncased,1.5,2.138203,2.186214,5.624479,"[2.587324, 0.9498999, 4.7450647, 3.0460522, 0....",2.587324,0.949900,4.745065,3.046052,0.648108,2.517560,0.809487


### Text Scale Ablation — Overall MAE Summary

In [35]:
image_only_row = next(r for r in results if r.get("text_model") in [None, "none"])
baseline_mae   = image_only_row["test_mae"]

scale_df["delta_vs_baseline"] = (baseline_mae - scale_df["test_mae"]).round(4)
scale_df["pct_improvement"]   = ((scale_df["delta_vs_baseline"] / baseline_mae) * 100).round(2)

print(f"Image-only baseline test MAE : {baseline_mae:.4f}\n")
print(f"Best config : {BEST_TEXT_COL}  +  {BEST_TEXT_MODEL}\n")

display_cols = ["text_scale", "val_mae", "test_mae", "test_rmse", "delta_vs_baseline", "pct_improvement"]
scale_df[display_cols]

Image-only baseline test MAE : 2.8121

Best config : text_qwen3_4b_clean  +  distilbert-base-uncased



,text_scale,val_mae,test_mae,test_rmse,delta_vs_baseline,pct_improvement
0,0.3,2.461822,2.544542,6.554314,0.2675,9.51
1,0.5,2.226678,2.325318,5.892409,0.4868,17.31
2,0.7,2.237237,2.262457,5.832594,0.5496,19.54
3,1.0,2.236047,2.269548,5.808817,0.5425,19.29
4,1.5,2.138203,2.186214,5.624479,0.6259,22.26


### Text Scale Ablation — Class-wise MAE

In [36]:
class_mae_cols = [f"mae_{cls}" for cls in TARGET_COLS]
class_scale_df = scale_df[["text_scale"] + class_mae_cols].copy()
class_scale_df.columns = ["text_scale"] + TARGET_COLS

baseline_class_mae = np.array([image_only_row[f"mae_{cls}"] for cls in TARGET_COLS])

delta_rows = []
for _, row in class_scale_df.iterrows():
    row_mae = np.array([row[cls] for cls in TARGET_COLS])
    delta_rows.append(
        {"text_scale": row["text_scale"],
         **{cls: round(baseline_class_mae[i] - row_mae[i], 4) for i, cls in enumerate(TARGET_COLS)}}
    )

delta_class_df = pd.DataFrame(delta_rows)

print("Class-wise MAE per text scale")
print(class_scale_df.to_string(index=False))

print("\nClass-wise improvement vs image-only baseline (positive = better)")
print(delta_class_df.to_string(index=False))

Class-wise MAE per text scale
 text_scale     Tree    Shrub    Grass     Crop  Built-up   Barren    Water
        0.3 2.915863 0.948505 5.923044 3.760775  0.644541 2.752447 0.866620
        0.5 2.904696 0.956013 5.134521 3.264056  0.703027 2.607940 0.706970
        0.7 2.558938 0.954104 4.950800 3.263929  0.722657 2.481685 0.905083
        1.0 2.695654 0.951909 4.841776 3.301911  0.770812 2.544267 0.780507
        1.5 2.587324 0.949900 4.745065 3.046052  0.648108 2.517560 0.809487

Class-wise improvement vs image-only baseline (positive = better)
 text_scale    Tree  Shrub  Grass   Crop  Built-up  Barren  Water
        0.3 -0.0879 0.0171 0.4437 1.2697    0.2267 -0.1495 0.1531
        0.5 -0.0768 0.0096 1.2322 1.7664    0.1682 -0.0050 0.3128
        0.7  0.2690 0.0115 1.4159 1.7665    0.1485  0.1213 0.1146
        1.0  0.1323 0.0137 1.5249 1.7285    0.1004  0.0587 0.2392
        1.5  0.2406 0.0157 1.6217 1.9844    0.2231  0.0854 0.2102


## Text Scale Ablation

This section evaluates the effect of text_scale on model performance using
the best configuration identified from the main experiments (text_qwen3_4b_clean
with DistilBERT), with the image-only baseline test MAE of 2.812.

Overall MAE across scales shows a clear trend of improvement as scale
increases, with scale 1.5 achieving the best test MAE of 2.186 and the highest
percentage improvement of 22.26% over the baseline. Scale 0.3 provides the
weakest improvement at 9.51%, confirming that stronger text contribution
consistently benefits DeiT Tiny across all tested scales.

Class-wise MAE across scales shows that scale 0.7 achieves the best Grass
MAE (4.951) while scale 1.5 performs best on Crop (3.046) and Tree (2.587).
Higher scales generally reduce errors on the dominant classes.

Class-wise improvement over the image only baseline confirms that Grass and Crop benefit the most, with improvements of up to 1.62
and 1.98 respectively at scale 1.5. Barren and Water also improve consistently
at higher scales. Built-up shows small but consistent gains across all scales.
Tree shows slight degradation at low scales (0.3, 0.5) but recovers and improves
at scales 0.7 and above. Shrub remains largely unaffected throughout.

Overall, DeiT Tiny benefits from higher text scales more than SwinV2 and MaxViT,
consistent with its weaker visual backbone relying more heavily on textual
information to compensate.

## Summary of All DeiT-Tiny Experiments

In [37]:
scale_experiment_ids = {id(r) for r in scale_results}

all_results_df = pd.DataFrame(results + scale_results).copy()
all_results_df["_source"] = (
    ["results"] * len(results) + ["scale_sweep"] * len(scale_results)
)

all_results_df["experiment_type"] = all_results_df.apply(
    lambda r: "image_only"   if pd.isna(r.get("text_model")) or r.get("text_model") in [None, "none"]
    else ("scale_sweep"      if r["_source"] == "scale_sweep"
    else  "multimodal_0.7"),
    axis=1
)

all_results_df = all_results_df.drop(columns=["_source"])

display_cols = [
    "experiment_type", "text_column", "text_model",
    "text_scale", "val_mae", "test_mae", "test_rmse"
]

all_results_df[display_cols].sort_values(
    ["experiment_type", "test_mae"]
).reset_index(drop=True)

,experiment_type,text_column,text_model,text_scale,val_mae,test_mae,test_rmse
0,image_only,NaN,none,NaN,NaN,2.812086,7.659360
1,multimodal_0.7,text_qwen3_4b_clean,distilbert-base-uncased,0.7,2.285313,2.323391,6.055197
2,multimodal_0.7,text_qwen3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.313581,2.379557,6.205260
3,multimodal_0.7,text_qwen3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.376651,2.397682,6.197832
4,multimodal_0.7,hybrid_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.481461,2.518511,6.544519
5,multimodal_0.7,hybrid_gemma3_4b_clean,distilbert-base-uncased,0.7,2.482271,2.591328,6.873420
6,multimodal_0.7,hybrid_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.499080,2.697665,6.951913
7,multimodal_0.7,vision_gemma3_4b_clean,RemoteCLIP ViT-B-32,0.7,2.726500,2.855073,7.644063
8,multimodal_0.7,vision_gemma3_4b_clean,sentence-transformers/all-MiniLM-L6-v2,0.7,2.928293,3.090967,8.295122
9,multimodal_0.7,vision_gemma3_4b_clean,distilbert-base-uncased,0.7,2.975283,3.101354,7.963865


In [38]:
# Best configuration overall — across all experiments including scale sweep
all_multimodal = all_results_df[all_results_df["experiment_type"] != "image_only"]
best_overall = all_multimodal.loc[all_multimodal["val_mae"].idxmin()]

print("Best overall configuration (by validation MAE)")
print(f"  Experiment type : {best_overall['experiment_type']}")
print(f"  Text column     : {best_overall['text_column']}")
print(f"  Text model      : {best_overall['text_model']}")
print(f"  Text scale      : {best_overall['text_scale']}")
print(f"  Val MAE         : {best_overall['val_mae']:.4f}")
print(f"  Test MAE        : {best_overall['test_mae']:.4f}")
print(f"  Test RMSE       : {best_overall['test_rmse']:.4f}")

Best overall configuration (by validation MAE)
  Experiment type : scale_sweep
  Text column     : text_qwen3_4b_clean
  Text model      : distilbert-base-uncased
  Text scale      : 1.5
  Val MAE         : 2.1382
  Test MAE        : 2.1862
  Test RMSE       : 5.6245


The table consolidates all experiment types: image only baseline, multimodal
at fixed scale 0.7, and scale sweep results.

The image only baseline achieves a test MAE of 2.812. All text only and hybrid
multimodal configurations improve over the baseline, while vision based captions
remain at or above it across all encoders.

The best overall configuration across all experiments is text_qwen3_4b_clean
with DistilBERT at scale 1.5, achieving a validation MAE of 2.138 and test MAE
of 2.186. The scale sweep confirms that higher text scales consistently benefit
DeiT Tiny, with scale 1.5 outperforming the default 0.7 (test MAE: 2.262) by
a meaningful margin. This contrasts with SwinV2 where scale 0.7 was optimal,
further suggesting that the weaker DeiT Tiny backbone relies more on textual
contribution to compensate for its limited visual capacity.

Among the fixed scale 0.7 experiments, text only captions rank 1st through 3rd
regardless of encoder, confirming that caption quality matters more than encoder
choice for DeiT Tiny.

Overall, DeiT Tiny shows the largest absolute improvement from multimodal fusion
across all three backbones, with the best configuration reducing test MAE from
2.812 to 2.186 which is a 22.3% improvement.

In [39]:
BACKBONE_NAME = CONFIG["model_name"]
save_path = os.path.join(CONFIG["predictions_dir"], f"all_results_{BACKBONE_NAME}.csv")
all_results_df.to_csv(save_path, index=False)
print("Saved results to Drive")

Saved results to Drive
